# 📊 Strongbox Parser - Google Colab Edition

**Convert Strongbox Excel files to Audit Sight format - No installation required!**

## 🚀 Instructions:
1. **Click Runtime → Run all** (or press Ctrl+F9)
2. **Wait for setup** to complete (about 30 seconds)
3. **Upload your file** when prompted
4. **Download** the processed result

**Features:**
- ✅ Processes multiple TXN-FY sheets with real transaction data
- ✅ Auto-detects date ranges from TB data
- ✅ Creates Comparative Trial Balance from your accounts
- ✅ Generates Journal Entries & Lines from your transactions
- ✅ Professional Excel formatting
- ✅ Data cleaning and Unicode handling
- ✅ Complete with all required tabs (Instructions, Banking, etc.)

**Team-friendly: Share this link with anyone who needs to process Strongbox files!**

In [ ]:
#@title 🔧 Setup (Run this first)
print('🔧 Installing required packages...')
!pip install -q openpyxl python-dateutil xlsxwriter
print('✅ Setup complete! Ready to process files.')

In [ ]:
#@title 📚 Import Libraries
import pandas as pd
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
import calendar
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from copy import copy
import math
import sys
from google.colab import files
import io
import re
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded successfully!')

In [ ]:
#@title 🏗️ Load Complete Strongbox Parserclass StrongboxParserColab:    def __init__(self):        self.source_file = None        self.start_date = None        self.end_date = None        self.begin_balance_date = None        self.source_data = {}        self.template_data = {}        self.output_filename = None        self.date_columns = {}    def print_and_log(self, message):        print(message)    def update_status(self, message, progress=None):        if progress:            print(f'[{progress:2d}%] {message}')        else:            print(message)    def determine_date_range(self):        self.update_status('Determining date range from TB sheet...', 10)                date_row = pd.read_excel(self.source_file, sheet_name='TB', header=None, nrows=1, skiprows=3)        date_row = date_row.iloc[0]                date_columns = {}        for col_idx, value in enumerate(date_row):            try:                if pd.notna(value):                    if isinstance(value, str):                        for fmt in ['%m/%d/%Y', '%Y-%m-%d', '%m-%d-%Y', '%d-%b-%Y', '%d/%b/%Y']:                            try:                                date = datetime.strptime(value, fmt)                                date_columns[date] = col_idx                                break                            except ValueError:                                continue                    else:                        date = pd.to_datetime(value)                        date_columns[date] = col_idx            except Exception:                continue                if not date_columns:            raise Exception('No valid dates found in TB sheet row 4')                tb_dates = sorted(date_columns.keys())        if len(tb_dates) < 2:            raise Exception('Need at least two dates in TB sheet')                self.begin_balance_date = tb_dates[0]        self.start_date = tb_dates[0] + relativedelta(days=1)        self.end_date = tb_dates[-1]        self.date_columns = date_columns                self.print_and_log(f'📅 Date range: {self.start_date.strftime("%Y-%m-%d")} to {self.end_date.strftime("%Y-%m-%d")}')        self.print_and_log(f'📅 Date columns mapping: {[(d.strftime("%Y-%m-%d"), idx) for d, idx in sorted(date_columns.items())]}')        return date_columns    def load_source_data(self):        self.update_status('Loading transaction data...', 20)                excel_file = pd.ExcelFile(self.source_file)        txn_sheets = [s for s in excel_file.sheet_names if s.startswith('TXN-FY')]                self.print_and_log(f'📊 Found {len(txn_sheets)} transaction sheets: {txn_sheets}')                for sheet_name in txn_sheets:            try:                df = pd.read_excel(self.source_file, sheet_name=sheet_name)                self.print_and_log(f'📋 {sheet_name}: {len(df)} rows loaded')                                # Convert date columns using the correct column names                for col in ['Transaction Date', 'Fiscal Month']:                    if col in df.columns:                        df[col] = pd.to_datetime(df[col], errors='coerce')                                # Convert numeric columns using the correct column names                for col in ['Debit', 'Credit']:                    if col in df.columns:                        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)                                # Filter by Fiscal Month date range (the original script uses Fiscal Month for filtering)                if self.start_date is not None and self.end_date is not None:                    if 'Fiscal Month' in df.columns:                        mask = (df['Fiscal Month'] >= self.begin_balance_date) & (df['Fiscal Month'] <= self.end_date)                        df_filtered = df[mask]                        self.print_and_log(f'📅 {sheet_name}: {len(df_filtered)} transactions in date range')                        df = df_filtered                                if not df.empty:                    self.source_data[sheet_name] = df                    self.print_and_log(f'✅ {sheet_name}: {len(df)} transactions loaded')                            except Exception as e:                self.print_and_log(f'⚠️ Error loading {sheet_name}: {str(e)}')        # Load TB-DATA sheet for balance information        self.print_and_log("\nLoading TB-DATA sheet...")        self.update_status("Loading TB-DATA sheet...", 25)        try:            tb_data_df = pd.read_excel(                self.source_file,                sheet_name='TB-DATA',                engine='openpyxl',                na_filter=False,                keep_default_na=False            )            self.source_data['TB-DATA'] = tb_data_df            self.print_and_log(f"✅ Successfully loaded TB-DATA sheet with {len(tb_data_df)} rows")            self.print_and_log(f"TB-DATA columns: {tb_data_df.columns.tolist()}")                        # Show first few rows for debugging            self.print_and_log("\nFirst few rows of TB-DATA:")            self.print_and_log(tb_data_df.head().to_string())                    except Exception as e:            self.print_and_log(f"⚠️ Warning: Could not load TB-DATA sheet: {str(e)}")            self.print_and_log("Will use default balance values (0) if TB-DATA is not available")            self.source_data['TB-DATA'] = None    def _extract_balances_from_tb_data(self, account_id, begin_date, end_date):        """Extract beginning and ending balances from TB-DATA sheet for a specific account"""        if 'TB-DATA' not in self.source_data or self.source_data['TB-DATA'] is None:            return 0.0, 0.0                tb_data = self.source_data['TB-DATA']                # Debug: Show the first few rows and column structure for the first account only        if not hasattr(self, '_tb_data_debug_shown'):            self.print_and_log("\nDEBUG: TB-DATA structure for balance extraction:")            self.print_and_log(f"TB-DATA columns: {tb_data.columns.tolist()}")            self.print_and_log("First 3 rows of TB-DATA:")            for i in range(min(3, len(tb_data))):                row_data = []                for j in range(min(8, len(tb_data.columns))):  # Show first 8 columns                    row_data.append(f"Col{j}: {tb_data.iloc[i, j]}")                self.print_and_log(f"Row {i}: {', '.join(row_data)}")            self._tb_data_debug_shown = True                # Find the account in TB-DATA        # Try different possible account ID column positions        account_data = None        account_col_found = None                # Check columns B through D (indices 1-3) for Account ID        for col_idx in [1, 2, 3]:            try:                if col_idx < len(tb_data.columns):                    matching_rows = tb_data[tb_data.iloc[:, col_idx].astype(str) == str(account_id)]                    if not matching_rows.empty:                        account_data = matching_rows                        account_col_found = col_idx                        break            except Exception:                continue                if account_data is None or account_data.empty:            # Account not found in TB-DATA            return 0.0, 0.0                begin_balance = 0.0        end_balance = 0.0        matches_found = 0                # Find balances matching our target dates        for _, row in account_data.iterrows():            try:                # Column A (index 0) should contain fiscal month                fiscal_month_val = row.iloc[0]                if pd.isna(fiscal_month_val):                    continue                                # Try to parse the fiscal month date                fiscal_month = pd.to_datetime(fiscal_month_val)                                # Column E (index 4) is Starting Account Balance                  starting_balance_val = row.iloc[4] if len(row) > 4 else 0.0                starting_balance = 0.0                if pd.notna(starting_balance_val):                    try:                        starting_balance = float(starting_balance_val)                    except (ValueError, TypeError):                        starting_balance = 0.0                                # Column G (index 6) is Ending Account Balance                ending_balance_val = row.iloc[6] if len(row) > 6 else 0.0                ending_balance = 0.0                if pd.notna(ending_balance_val):                    try:                        ending_balance = float(ending_balance_val)                    except (ValueError, TypeError):                        ending_balance = 0.0                                # Check if fiscal month matches our target dates                # Beginning Balance uses Ending Account Balance for begin date                if fiscal_month.date() == begin_date.date():                    begin_balance = ending_balance  # Changed from starting_balance to ending_balance                    matches_found += 1                                    if fiscal_month.date() == end_date.date():                    end_balance = ending_balance                    matches_found += 1                                except Exception as e:                continue                # Debug for first few accounts        if matches_found > 0 and not hasattr(self, f'_balance_debug_{account_id}'):            self.print_and_log(f"Found balances for account {account_id}: Begin={begin_balance}, End={end_balance}")            setattr(self, f'_balance_debug_{account_id}', True)                return begin_balance, end_balance    def create_trial_balance(self):        """Create Comparative Trial Balances tab (matches original output format)"""        self.update_status('Creating trial balance...', 70)        tb_data = self.source_data['TB']                # Filter out header rows (matches original logic)        tb_data = tb_data[~((tb_data['Account Id'].str.lower() == 'account id') |                            (tb_data['Account Id'].str.lower() == 'account'))]                # Create DataFrame with exact column names from original, but we'll update balances from TB-DATA        trial_balance = pd.DataFrame({            'Account ID': tb_data['Account Id'],            'Account Name': tb_data['Account Name'],            'Beginning Balance \\n(Prior Period Balance)': 0.0,  # Will be populated from TB-DATA            'Ending Balance': 0.0,  # Will be populated from TB-DATA            'Account Type \\n(see Mapping Categories tab)': '',            'Account Mapping \\n(see Mapping Categories tab)': '',            'Account Description': tb_data['Financial Statement Classification']        })                # Update balances from TB-DATA sheet        self.print_and_log("\nUpdating balances from TB-DATA sheet...")        self.update_status("Updating balances from TB-DATA...", 72)                updated_accounts = 0        total_begin_from_tbdata = 0.0        total_end_from_tbdata = 0.0                for idx, row in trial_balance.iterrows():            account_id = row['Account ID']            begin_balance, end_balance = self._extract_balances_from_tb_data(                account_id, self.begin_balance_date, self.end_date            )                        trial_balance.at[idx, 'Beginning Balance \\n(Prior Period Balance)'] = begin_balance            trial_balance.at[idx, 'Ending Balance'] = end_balance                        total_begin_from_tbdata += begin_balance            total_end_from_tbdata += end_balance                        if begin_balance != 0.0 or end_balance != 0.0:                updated_accounts += 1                self.print_and_log(f"Updated balances for {updated_accounts} accounts from TB-DATA")        self.print_and_log(f"Total balances extracted from TB-DATA: Begin={total_begin_from_tbdata}, End={total_end_from_tbdata}")                # Apply account type classification        trial_balance['Account Type \\n(see Mapping Categories tab)'] = trial_balance['Account Description'].apply(self.determine_account_type)                # Debug: show sample of trial balance        self.print_and_log(f'Trial Balance sample columns: {trial_balance.columns.tolist()}')        if len(trial_balance) > 0:            self.print_and_log(f'Sample TB row: {trial_balance.iloc[0].to_dict()}')                # Calculate sum of beginning and ending balances        begin_sum = trial_balance['Beginning Balance \\n(Prior Period Balance)'].sum()        end_sum = trial_balance['Ending Balance'].sum()        self.print_and_log(f"Sum of Beginning Balances: {begin_sum}")        self.print_and_log(f"Sum of Ending Balances: {end_sum}")                # Show a few sample balances to verify they're being set correctly        self.print_and_log("\nSample balance values from trial balance:")        for idx in range(min(5, len(trial_balance))):            account_id = trial_balance.at[idx, 'Account ID']            begin_bal = trial_balance.at[idx, 'Beginning Balance \\n(Prior Period Balance)']            end_bal = trial_balance.at[idx, 'Ending Balance']            self.print_and_log(f"Account {account_id}: Begin={begin_bal}, End={end_bal}")                # Remove rows with empty Account IDs        trial_balance = trial_balance[trial_balance['Account ID'].notna() & (trial_balance['Account ID'] != '')]        trial_balance = trial_balance.reset_index(drop=True)                self.print_and_log(f'✅ Trial balance created: {len(trial_balance)} accounts')        return trial_balance    def create_journal_entries(self):        """Create Journal Entries & Lines tab (matches original output format)"""        self.update_status('Creating journal entries...', 60)                # Get transaction sheets        transaction_sheets = {k: v for k, v in self.source_data.items() if k.startswith('TXN-FY')}        self.print_and_log(f'Found sheets: {list(transaction_sheets.keys())}')                processed_sheets = []        for sheet_name, df in transaction_sheets.items():            try:                self.print_and_log(f'Processing sheet: {sheet_name}')                df_copy = df.copy()                                # Convert columns to appropriate types (using original column names)                df_copy['Transaction Id'] = df_copy['Transaction Id'].astype(str)                df_copy['Memo'] = df_copy['Memo'].fillna('')                df_copy['Doc/Ref No'] = df_copy['Doc/Ref No'].fillna('')                df_copy['Account Id'] = df_copy['Account Id'].astype(str)                                # Handle new columns (with fallback if columns don't exist)                if 'Transaction Type' in df_copy.columns:                    df_copy['Transaction Type'] = df_copy['Transaction Type'].fillna('')                else:                    df_copy['Transaction Type'] = ''                    self.print_and_log(f'Warning: Transaction Type column not found in {sheet_name}, using empty values')                                if 'Relationship Name' in df_copy.columns:                    df_copy['Relationship Name'] = df_copy['Relationship Name'].fillna('')                else:                    df_copy['Relationship Name'] = ''                    self.print_and_log(f'Warning: Relationship Name column not found in {sheet_name}, using empty values')                                # Convert numeric columns                df_copy['Debit'] = pd.to_numeric(df_copy['Debit'], errors='coerce').fillna(0)                df_copy['Credit'] = pd.to_numeric(df_copy['Credit'], errors='coerce').fillna(0)                                # Create the required columns with exact names and new column positions                processed_df = pd.DataFrame({                    'Journal ID': df_copy['Transaction Id'],                    'Type': df_copy['Transaction Type'],                    'Journal Entry Description': df_copy['Doc/Ref No'],                    'Posted Date': df_copy['Transaction Date'],                    'Account ID': df_copy['Account Id'],                    'Journal Line Description': df_copy['Memo'],                    'Name': df_copy['Relationship Name'],                    'Debit Amount': df_copy['Debit'],                    'Credit Amount': df_copy['Credit']                })                                processed_sheets.append(processed_df)                self.print_and_log(f'Successfully processed sheet: {sheet_name}')                            except Exception as e:                self.print_and_log(f'Error processing sheet {sheet_name}: {str(e)}')                continue                # Combine all processed sheets        if not processed_sheets:            raise Exception('No sheets were successfully processed')                journal_entries = pd.concat(processed_sheets, ignore_index=True)        self.print_and_log(f'✅ Journal entries created: {len(journal_entries)} entries')        return journal_entries    def create_other_sheets(self, workbook, styles):        """Create and format all other required sheets"""        self.print_and_log('Creating additional required sheets...')                # Create Instructions sheet        instructions_sheet = workbook.create_sheet('Instructions')        instructions_sheet.append(['Strongbox Parser Instructions'])        instructions_sheet.append(['This file was processed by Strongbox Parser for Audit Sight import.'])        instructions_sheet.append([''])        instructions_sheet.append(['Main Tabs:'])        instructions_sheet.append(['- Comparative Trial Balances: Account balances and classifications'])        instructions_sheet.append(['- Journal Entries & Lines: All transaction detail'])        instructions_sheet.append(['- Banking Accts: Banking account setup (if applicable)'])        instructions_sheet.append(['- Banking Txn: Banking transactions (if applicable)'])                # Create Data Validation Tests sheet        validation_sheet = workbook.create_sheet('Data Validation Tests')        validation_sheet.append(['Data Validation Tests'])        validation_sheet.append(['This sheet can be used for data validation checks.'])                # Create Notes sheet        notes_sheet = workbook.create_sheet('Notes')        notes_sheet.append(['Notes and Comments'])        notes_sheet.append(['Add any notes about this import here.'])                # Create Banking Accts sheet with specific formatting        banking_accts_sheet = workbook.create_sheet('Banking Accts')                # Add headers for Banking Accts        banking_accts_sheet.append(['Required', 'Required', 'Optional', 'Optional'])        banking_accts_sheet.append(['Account Number', 'Account Name', 'Institution', 'Currency'])                # Style row 1 headers for Banking Accts        for col in range(1, 3):  # Columns A-B (Required)            cell = banking_accts_sheet.cell(row=1, column=col)            cell.font = styles['header_font']            cell.fill = styles['blue_fill']            cell.alignment = styles['center_alignment']                for col in range(3, 5):  # Columns C-D (Optional)            cell = banking_accts_sheet.cell(row=1, column=col)            cell.font = styles['header_font']            cell.fill = styles['gray_fill']            cell.alignment = styles['center_alignment']                # Style row 2 headers for Banking Accts        for col in range(1, 5):  # All columns A-D            cell = banking_accts_sheet.cell(row=2, column=col)            cell.font = styles['header_font']            cell.fill = styles['dark_blue_fill']            cell.alignment = styles['center_alignment']                # Create Banking Txn sheet with specific formatting        banking_txn_sheet = workbook.create_sheet('Banking Txn')                # Add headers for Banking Txn        banking_txn_sheet.append(['Required', 'Required', 'Required', 'Required'])        banking_txn_sheet.append(['Posted Date', 'Description', 'Amount', 'Account Number'])                # Style row 1 headers for Banking Txn (all Required)        for col in range(1, 5):  # Columns A-D (all Required)            cell = banking_txn_sheet.cell(row=1, column=col)            cell.font = styles['header_font']            cell.fill = styles['blue_fill']            cell.alignment = styles['center_alignment']                # Style row 2 headers for Banking Txn        for col in range(1, 5):  # All columns A-D            cell = banking_txn_sheet.cell(row=2, column=col)            cell.font = styles['header_font']            cell.fill = styles['dark_blue_fill']            cell.alignment = styles['center_alignment']                # Create Mapping Categories sheet        mapping_sheet = workbook.create_sheet('Mapping Categories')        mapping_sheet.append(['Account Type Mapping Categories'])        mapping_sheet.append([''])        mapping_sheet.append(['Valid Account Types:'])        mapping_sheet.append(['Asset'])        mapping_sheet.append(['Liability'])        mapping_sheet.append(['Equity'])        mapping_sheet.append(['Revenue'])        mapping_sheet.append(['Expense'])    def create_output_file(self, trial_balance, journal_entries):        """Create Excel output file with professional formatting (matches original)"""        self.update_status('Creating Excel output...', 80)                start_str = self.start_date.strftime('%Y%m%d')        end_str = self.end_date.strftime('%Y%m%d')        self.output_filename = f'Processed_Strongbox_{start_str}_{end_str}.xlsx'                workbook = openpyxl.Workbook()                # Remove default sheet        if 'Sheet' in workbook.sheetnames:            workbook.remove(workbook['Sheet'])                # Create styles (matches original)        header_font = Font(name='Arial', size=12, bold=True, color='FFFFFF')        blue_fill = PatternFill(start_color='0070C0', end_color='0070C0', fill_type='solid')        gray_fill = PatternFill(start_color='999999', end_color='999999', fill_type='solid')        dark_blue_fill = PatternFill(start_color='002060', end_color='002060', fill_type='solid')        center_alignment = Alignment(horizontal='center', vertical='center')                styles = {            'header_font': header_font,            'blue_fill': blue_fill,            'gray_fill': gray_fill,            'dark_blue_fill': dark_blue_fill,            'center_alignment': center_alignment        }                # Create Trial Balance sheet FIRST        tb_sheet = workbook.create_sheet('Comparative Trial Balances')                # Set column widths for trial balance        tb_sheet.column_dimensions['A'].width = 17.9        tb_sheet.column_dimensions['B'].width = 40.0        tb_sheet.column_dimensions['C'].width = 20.0        tb_sheet.column_dimensions['D'].width = 15.7        tb_sheet.column_dimensions['E'].width = 20.0        tb_sheet.column_dimensions['F'].width = 35.7        tb_sheet.column_dimensions['G'].width = 25.7                # Write trial balance headers and data        tb_headers = list(trial_balance.columns)        for col_idx, header in enumerate(tb_headers, 1):            cell = tb_sheet.cell(row=1, column=col_idx, value=header)            cell.font = header_font            cell.fill = dark_blue_fill            cell.alignment = center_alignment                # Write trial balance data        for row_idx, (_, row) in enumerate(trial_balance.iterrows(), 2):            for col_idx, value in enumerate(row, 1):                tb_sheet.cell(row=row_idx, column=col_idx, value=value)                # Create Journal Entries sheet SECOND        je_sheet = workbook.create_sheet('Journal Entries & Lines')                # Set column widths for journal entries (updated for 9 columns)        je_sheet.column_dimensions['A'].width = 14.3  # Journal ID        je_sheet.column_dimensions['B'].width = 15.0  # Type        je_sheet.column_dimensions['C'].width = 48.6  # Journal Entry Description        je_sheet.column_dimensions['D'].width = 15.7  # Posted Date        je_sheet.column_dimensions['E'].width = 20.0  # Account ID        je_sheet.column_dimensions['F'].width = 35.7  # Journal Line Description        je_sheet.column_dimensions['G'].width = 25.0  # Name        je_sheet.column_dimensions['H'].width = 15.7  # Debit Amount        je_sheet.column_dimensions['I'].width = 15.7  # Credit Amount                # Write journal entries headers with Required/Optional row (updated for new columns)        je_sheet.append(['Required', 'Optional', 'Optional', 'Required', 'Required', 'Optional', 'Optional', 'Required', 'Required'])        je_sheet.append(list(journal_entries.columns))                # Style headers (updated column positions)        required_cols = [1, 4, 5, 8, 9]  # Columns A, D, E, H, I (Journal ID, Posted Date, Account ID, Debit, Credit)        optional_cols = [2, 3, 6, 7]     # Columns B, C, F, G (Type, Description, Line Description, Name)                for col in required_cols:            cell = je_sheet.cell(row=1, column=col)            cell.font = header_font            cell.fill = blue_fill            cell.alignment = center_alignment                for col in optional_cols:            cell = je_sheet.cell(row=1, column=col)            cell.font = header_font            cell.fill = gray_fill            cell.alignment = center_alignment                # Style column name headers (updated for 9 columns)        for col in range(1, 10):            cell = je_sheet.cell(row=2, column=col)            cell.font = header_font            cell.fill = dark_blue_fill            cell.alignment = center_alignment                # Write journal entries data        for row_idx, (_, row) in enumerate(journal_entries.iterrows(), 3):            for col_idx, value in enumerate(row, 1):                cell = je_sheet.cell(row=row_idx, column=col_idx, value=value)                # Apply date formatting to Posted Date column (now column D)                if col_idx == 4:  # Posted Date column (was 3, now 4)                    cell.number_format = 'M/D/YYYY'                # Create all other required sheets AFTER main data sheets        self.create_other_sheets(workbook, styles)                workbook.save(self.output_filename)        workbook.close()                self.print_and_log(f'✅ Output file created: {self.output_filename}')        return self.output_filename    def run(self):        try:            self.print_and_log('🚀 Starting Strongbox Parser...')                        # First determine date range from TB sheet            self.determine_date_range()                        # Then load transaction data with date filtering            self.load_source_data()                        # Load trial balance data using openpyxl (like original)            tb_data = self.load_trial_balance_data()            self.source_data['TB'] = tb_data                        # Create outputs from real data            trial_balance = self.create_trial_balance()            journal_entries = self.create_journal_entries()                        # Create output file            output_file = self.create_output_file(trial_balance, journal_entries)                        self.update_status('Processing complete!', 100)            self.print_and_log('\n🎉 SUCCESS! Processing completed!')                        return output_file                    except Exception as e:            self.print_and_log(f'\n❌ Error: {str(e)}')            raiseprint('✅ Complete Strongbox Parser loaded and ready!')

In [ ]:
#@title 📁 Upload Your Strongbox Excel File

print('📁 UPLOAD YOUR STRONGBOX EXCEL FILE')
print('=' * 40)
print('Requirements:')
print('  ✓ Excel file (.xlsx format)')
print('  ✓ TB sheet with trial balance data')
print('  ✓ TXN-FY sheets with transactions')
print('  ✓ Dates in row 4 of TB sheet')
print()
print('Click Choose Files below and select your file:')

uploaded = files.upload()

if uploaded:
    source_filename = list(uploaded.keys())[0]
    file_size = len(uploaded[source_filename])
    print(f'\n✅ SUCCESS! File uploaded:')
    print(f'   📄 Name: {source_filename}')
    print(f'   📊 Size: {file_size:,} bytes ({file_size/1024/1024:.1f} MB)')
    print(f'\n🔄 Ready to process! Run the next cell.')
else:
    print('\n❌ No file uploaded. Please try again.')
    source_filename = None

In [ ]:
#@title 🔄 Process Your File

if 'source_filename' in globals() and source_filename:
    print('🔄 PROCESSING YOUR STRONGBOX FILE')
    print('=' * 35)
    print(f'File: {source_filename}')
    print()
    
    # Initialize and run parser
    parser = StrongboxParserColab()
    parser.source_file = source_filename
    
    try:
        output_file = parser.run()
        
        print('\n' + '=' * 50)
        print('🎉 SUCCESS! YOUR FILE HAS BEEN PROCESSED!')
        print('=' * 50)
        print(f'✅ Output file: {output_file}')
        print('\n📋 Your file contains:')
        print('   📊 Comparative Trial Balances (Your Real Data)')
        print('   📝 Journal Entries & Lines (Your Real Transactions)')
        print('   📋 Instructions, Banking Accts, Banking Txn, etc.')
        print('   🎨 Professional formatting')
        print('   🧹 Data cleaning applied')
        print('\n📥 Ready for download! Run the next cell.')
        
    except Exception as e:
        print('\n' + '=' * 40)
        print('❌ PROCESSING ERROR')
        print('=' * 40)
        print(f'Error: {str(e)}')
        print('\n🔍 Please check:')
        print('  • File has TB sheet with trial balance')
        print('  • TB sheet has dates in row 4')
        print('  • File has TXN-FY sheets with transactions')
        print('  • File is not password protected')
        print('  • File format is .xlsx (not .xls)')
        output_file = None
        
else:
    print('⚠️ Please upload a file first by running the cell above.')
    output_file = None

In [ ]:
#@title 📥 Download Your Processed File

if 'output_file' in globals() and output_file:
    print('📥 DOWNLOADING YOUR PROCESSED FILE')
    print('=' * 38)
    print(f'File: {output_file}')
    print('\nThe file will download to your browser\'s Downloads folder.')
    print()
    
    # Check if file exists before downloading
    import os
    if os.path.exists(output_file):
        # Download the file
        files.download(output_file)
        
        print('✅ Download started!')
        print('\n🎉 ALL DONE!')
        print('=' * 15)
        print('Your Audit Sight formatted file is ready!')
        print('\n💡 What you got:')
        print('  • Complete trial balance from your TB sheet')
        print('  • All journal entries from your TXN-FY sheets')
        print('  • Proper date filtering and account classification')
        print('  • Unicode cleaning and data validation')
        print('  • Professional Excel formatting')
        print('  • All required tabs for Audit Sight import')
        print('\n📊 Ready for Audit Sight import!')
    else:
        print('❌ Error: Output file not found!')
        print(f'Expected file: {output_file}')
    
else:
    print('⚠️ No file ready for download.')
    print('Please upload and process a file first.')